In [1]:
import pandas as pd

In [2]:
# split by tabs not commas
df = pd.read_csv('SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])

## Class imbalance

* What's the ratio of ham to spam?
* What's the level of imbalance? Is it critical?
* What are we going to do about this in the future?

In [3]:
df['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [4]:
df['label'].value_counts(normalize=True)

label
ham     0.865937
spam    0.134063
Name: proportion, dtype: float64

With 86.6% ham / 13.4% spam:

* We'll **not use SMOTE** (that's for more severe imbalance, and it can create synthetic text that doesn't make grammatical sense — SMOTE was designed for numeric/tabular features, not raw text).
* Instead we'll use **LogisticRegression(class_weight='balanced')**

## Are spam messages longer or shorter than ham?

In [5]:
# add length column
df['length'] = df['message'].apply(len)

In [6]:
df.groupby('label')['length'].mean()

label
ham      71.482487
spam    138.670683
Name: length, dtype: float64

In [7]:
df.groupby('label')['length'].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
ham,4825.0,71.482487,58.440652,2.0,33.0,52.0,93.0,910.0
spam,747.0,138.670683,28.873603,13.0,133.0,149.0,157.0,223.0


That 71-vs-139-character gap means length itself can be a **useful input feature**, separate from the words.

The spam message needs to contain to work: a hook, an offer, urgency, and usually a call-to-action (a phone number, a link, "reply STOP to unsubscribe" — required by SMS regulations).

In [8]:
# how much duplicates we have
df.duplicated().sum()

np.int64(403)

In [9]:
df[df.duplicated()].head()

,label,message,length
103,ham,As per your request 'Melle Melle (Oru Minnamin...,160
154,ham,As per your request 'Melle Melle (Oru Minnamin...,160
207,ham,"As I entered my cabin my PA said, '' Happy B'd...",156
223,ham,"Sorry, I'll call later",22
326,ham,No calls..messages..missed calls,32


Droping duplicates before splitting into train/test because we need the model correctly classify a message it has never seen before not recognizing concrete string

In [10]:
print("df.shape before removing duplicates:", df.shape)
df = df.drop_duplicates()
print("df.shape after removing duplicates:", df.shape)

df.shape before removing duplicates: (5572, 3)
df.shape after removing duplicates: (5169, 3)
